# Google Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/google_reviews.py`.

Covers:
1. API call – fetching reviews (Google Places API, max 5 per request)
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the API can't fetch

**Requirements:** `GOOGLE_MAPS_API_KEY` env var, Ollama running locally with `mistral:7b`.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [10]:
from src.sites import google_reviews as gr
from src.classification import classify_review, is_ollama_available, _parse_classification
import requests

#api_key = os.getenv('GOOGLE_MAPS_API_KEY')
if not api_key:
    raise RuntimeError('Set GOOGLE_MAPS_API_KEY before running this notebook')

print(f'Hotel: {gr.ANANEA_HOTEL}')
print(f'Google query: {gr.ANANEA_GOOGLE_QUERY}')
print(f'API key: {api_key[:1]}...')

Hotel: Ananea Castelo Suites Hotel
Google query: Ananea Castelo Suites Algarve, Portugal
API key: A...


## 1. API Call – Fetch Google Reviews

The Google Places API (New) returns a **maximum of 5 reviews** per request.
There is no pagination support for reviews.

In [4]:
# Fetch reviews for Ananea via Google Places Text Search
raw_reviews = gr.google_get_reviews(gr.ANANEA_GOOGLE_QUERY, api_key=api_key)
print(f'Reviews returned: {len(raw_reviews)}')
print()
for r in raw_reviews:
    rid = gr._extract_review_id(r)
    text = gr._extract_review_text(r)
    date = gr._extract_publish_date(r)
    rating = r.get('rating', 'N/A')
    author = r.get('authorAttribution', {}).get('displayName', 'Unknown')
    lang = r.get('originalText', {}).get('languageCode', r.get('text', {}).get('languageCode', '?'))
    print(f'  [{rid[:20]}...] {rating}/5 lang={lang} by {author} on {date}')
    print(f'    {text[:100]}...')
    print()

Reviews returned: 5

  [Ci9DQUlRQUNvZENodHlj...] 4/5 lang=en by Raj on 2025-11-11
    I booked this hotel on a whim as there were no jet 2 reviews on site. I did take a look at trip advi...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Lorena David on 2025-09-18
    The hotel was amazing! We stayed in the Maretta suite for a week and were very pleased with the room...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Elzbieta Krzyszton on 2025-09-09
    The best  hotel and staff, beautifull rooms and pool, the view, everything is great, i wish come bac...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Izzy Roberts on 2025-09-10
    I loved the violinist that plays in the hotel ! His music has a modern twist like Bridgerton 🎻🎻...

  [Ci9DQUlRQUNvZENodHlj...] 5/5 lang=en by Helen Turner on 2025-10-03
    Wonderful place to stay, lovely breakfast and fabulous staff...



In [5]:
# Inspect raw API response for first review
if raw_reviews:
    print(json.dumps(raw_reviews[0], indent=2, ensure_ascii=False))

{
  "name": "places/ChIJ5acUwUfNGg0RrUqtcamF0Y8/reviews/Ci9DQUlRQUNvZENodHljRjlvT2tkMlkwOVZTRmhJTmpOVFJrZzRTVkZsZGxCeE9FRRAB",
  "relativePublishTimeDescription": "4 months ago",
  "rating": 4,
  "text": {
    "text": "I booked this hotel on a whim as there were no jet 2 reviews on site. I did take a look at trip advisor which had a mixed bag and took the gamble anyway, definitely paid off!\n\nThe hotel is in a great location, they have a shuttle to the beach which I did not use, as I would say to three of the closest beaches it was less than a 20 walk. The room had a great view of the pool area, spacious enough for two people, smart tv which is a nice touch. Iron is available upon request.\n\nPros:\n\n- [ ] Great customer service all around, no issues which is hard to come by.\n- [ ] Plenty of sunbeds available and no rush to secure one like some other hotels\n- [ ] I liked that it was quiet, note this isn’t an adult only hotel, as during my stay there were a family with younger child

## 2. Ollama Classification – Test

In [6]:
# Check Ollama availability
ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    # List available models
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['mistral:7b']


In [7]:
# Classify a single review (pick the first with text)
test_review = next((r for r in raw_reviews if gr._extract_review_text(r)), None)
if test_review and ollama_ok:
    text = gr._extract_review_text(test_review)
    author = test_review.get('authorAttribution', {}).get('displayName', 'Unknown')
    print(f'Review by: {author}')
    print(f'Text: {text[:300]}...')
    print()
    topics = classify_review(text)
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

Review by: Raj
Text: I booked this hotel on a whim as there were no jet 2 reviews on site. I did take a look at trip advisor which had a mixed bag and took the gamble anyway, definitely paid off!

The hotel is in a great location, they have a shuttle to the beach which I did not use, as I would say to three of the close...

Classification result (8 topics):
  🟢 employees = positive
  🟢 commodities = positive
  🟢 comfort = positive
  🟢 meals = positive
  🟢 cleaning = positive
  🔴 commodities = negative
  🔴 comfort = negative
  🔴 meals = negative


In [7]:
# Classify ALL fetched reviews, save to JSON (overwrites existing reviews by ID)
from datetime import datetime

if ollama_ok:
    json_path = str(ROOT / 'data' / 'google_reviews.json')
    existing = gr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in raw_reviews:
        text = gr._extract_review_text(r)
        if not text:
            continue
        topics = classify_review(text)

        review_id = gr._extract_review_id(r)
        author_attr = r.get('authorAttribution', {})
        author_name = author_attr.get('displayName', '') if isinstance(author_attr, dict) else ''

        record = {
            'id': review_id,
            'hotel': gr.ANANEA_HOTEL,
            'source': 'google',
            'rating': r.get('rating'),
            'title': '',  # Google reviews have no title
            'text': text,
            'published_date': gr._extract_publish_date(r),
            'author_name': author_name,
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        stars = '\u2605' * (r.get('rating') or 0)
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{stars} [{action}] {author_name[:35]:35s} \u2192 {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    gr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available \u2013 skip classification')

★★★★ [added] Raj                                 → employees(pos), commodities(pos), comfort(pos), cleaning(pos), meals(pos), return(pos)
★★★★★ [added] Lorena David                        → employees(pos), meals(pos)
★★★★★ [added] Elzbieta Krzyszton                  → employees(pos), commodities(pos), comfort(pos), return(pos)
★★★★★ [added] Izzy Roberts                        → employees(pos)
★★★★★ [added] Helen Turner                        → employees(pos), meals(pos)

Saved 5 reviews to /home/laurabquintas/projects/reputation-analyzer/.claude/worktrees/vigilant-meitner/data/google_reviews.json


In [ ]:
# Debug: see the raw Ollama response for a specific review
# Change the index to test different reviews
DEBUG_INDEX = 0

if ollama_ok and raw_reviews:
    review = raw_reviews[DEBUG_INDEX]
    text = gr._extract_review_text(review)
    author = review.get('authorAttribution', {}).get('displayName', 'Unknown')
    print(f'Review by: {author}')
    print(f'Full text:\n{text}\n')
    print('--- Sending to Ollama ---')

    # Make raw request to see full response
    prompt = f"""You are a hotel review analyst. Read the review below carefully and identify ALL topics mentioned, even briefly.

TOPICS (use these exact keys):
- employees: any mention of staff, service, friendliness, helpfulness, reception, concierge, team, waiters, management
- commodities: amenities, facilities, pool, gym, spa, room features, wifi, parking, fridge, toiletries, TV, air conditioning, balcony
- comfort: room comfort, bed quality, noise, quiet, space, temperature, room size, mattress, pillow, decor, ambiance
- cleaning: cleanliness, hygiene, tidiness, housekeeping, spotless, dirty, stains, towels changed, room serviced
- quality_price: value for money, pricing, worth, cost, overpriced, good deal, expensive, cheap, affordable, half board value
- meals: food, breakfast, restaurant, dining, bar, drinks, buffet, dinner, lunch, cuisine, menu, chef, kitchen, snacks
- return: whether the guest would return, come back, visit again, recommend, revisit, not return, wouldn't go back

RULES:
1. You MUST check each topic one by one. Go through the review sentence by sentence.
2. A single review CAN have both positive AND negative for the SAME topic.
3. Even brief or indirect mentions count (e.g. "rooms were cleaned daily" = cleaning positive).
4. If a topic is described positively, mark it positive. If negatively, mark it negative.
5. Output ONLY a JSON array. No explanation, no markdown.

Now analyze this review:
\"\"\"{text}\"\"\"

JSON array:"""

    payload = {'model': 'mistral:7b', 'prompt': prompt, 'stream': False,
               'options': {'temperature': 0.1, 'num_predict': 512}}
    resp = requests.post('http://localhost:11434/api/generate', json=payload, timeout=60)
    raw = resp.json().get('response', '')
    print(f'Raw Ollama response:\n{raw}')
    print()
    parsed = _parse_classification(raw)
    print(f'Parsed: {json.dumps(parsed, indent=2)}')

## 3. Manual Review Input

Since the Google API returns only 5 reviews per request with no pagination,
you can manually add reviews below. Fill in the fields and run the cell.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'author_name': 'John D.',                      # reviewer name (used for ID)
    'rating': 5,                                    # 1-5
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
}
# ========================================

# Generate unique ID from name + date
id_seed = f"{manual_review['author_name']}_{manual_review['published_date']}_{manual_review['text'][:50]}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

record = {
    'id': review_id,
    'hotel': gr.ANANEA_HOTEL,
    'source': 'google',
    'rating': manual_review['rating'],
    'title': '',  # Google reviews have no title
    'text': manual_review['text'],
    'published_date': manual_review['published_date'],
    'author_name': manual_review['author_name'],
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
    'manual': True,
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text \u2013 skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'google_reviews.json')
existing = gr.load_reviews(json_path)

# Check for duplicates
existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'\u26a0\ufe0f  Review {record["id"]} already exists \u2013 skipping')
else:
    existing.append(record)
    gr.save_reviews(existing, json_path)
    print(f'\u2705 Saved! Total reviews: {len(existing)}')

## 4. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once. Useful for adding reviews from Google Maps
that aren't returned by the API.

In [9]:
import hashlib
from datetime import datetime

# Add multiple reviews at once
batch_reviews = [
    {
        'author_name': 'Maria S.',
        'rating': 4,
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
    },
    # Add more reviews here...
]
# Add multiple reviews at once

batch_reviews = [
    {
        'author_name': 'Seba Le',
        'rating': 4,
        'text': 'Sehr ruhiges Hotel. Eher nicht für Kinder geeignet, da es keine großen Unterhaltungsangebote gibt. Zimmer zur Straßenseite sind bei offenem Fenster recht laut. Quartos: 4.0, Serviço: 5.0, Localização: 4.0. Destaques do hotel: Ótimo valor. Quartos: Ohne Schränke. Atividades próximas: Alles was man möchte, jedoch meist mit Bus oder Auto erreichbar. Segurança: Sehr sicher. Acessibilidade a pé: Zwei schöne Strände – Castelo und Evaristo. Alimentos e bebidas: Super lecker.',
        'published_date': '2025-10-04',
    },
    {
        'author_name': 'Hans Naus',
        'rating': 5,
        'text': 'Fantastisch nieuw hotel. Goed ontbijt buffet, zeer goede bedden, kamer mooi met balkon en schitterend uitzicht. Diner buffet was ook zeer goed. Auto kon fijn geparkeerd worden. Al met al prima verblijf gehad. Quartos: 5.0, Serviço: 5.0, Localização: 5.0.',
        'published_date': '2025-10-04',
    },
    {
        'author_name': 'Heger',
        'rating': 5,
        'text': 'Quartos: 5.0, Serviço: 4.0, Localização: 5.0. Destaques do hotel: Vista fantástica · Sossegado · Alta tecnologia.',
        'published_date': '2025-10-04',
    },
    {
        'author_name': 'Ersen Unsurorlu',
        'rating': 4,
        'text': 'Quartos: 5.0, Serviço: 5.0, Localização: 3.0.',
        'published_date': '2025-10-04',
    },
    {
        'author_name': 'Pete',
        'rating': 4,
        'text': 'Ein sehr schönes, modernes Hotel mit Rooftop-Pool, von dem aus man den Sonnenuntergang beobachten kann. Stylische Elemente im gesamten Hotel. Sehr gutes Frühstück. Kostenloses Parken vor dem Hotel und Shuttle-Service zum nahegelegenen Strand. Photovoltaikanlage zur Unterstützung der Stromversorgung. Sehr freundliches Personal. Quartos: 4.0, Serviço: 5.0, Localização: 4.0. Destaques do hotel: Ótimo valor · Alta tecnologia.',
        'published_date': '2025-10-04',
    },
    {
        'author_name': 'Silke Engeln',
        'rating': 4,
        'text': 'Sehr schönes, schickes, neues Hotel. Sicherlich noch nicht alles eingespielt und an den ein oder anderen Stellen noch verbesserungsfähig. Wasser und Kaffee werden nicht immer regelmäßig aufgefüllt. Besonders hervorzuheben sind die Mitarbeiter im Restaurant, die sich täglich freundlich bemüht haben, obwohl es beim Frühstück oft zu wenig Plätze gab. Das Abendessen (Halbpension) war jeden Abend sehr lecker. In der Umgebung des Hotels gibt es jedoch wenig, daher ist ein Mietwagen sehr zu empfehlen. Quartos: 5.0, Serviço: 4.0, Localização: 3.0. Destaques do hotel: Vista fantástica · Sossegado.',
        'published_date': '2025-10-04',
    }
]

json_path = str(ROOT / 'data' / 'google_reviews.json')
existing = gr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['author_name']}_{mr['published_date']}_{mr['text'][:50]}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

    if rid in existing_ids:
        print(f'\u26a0\ufe0f  Skip duplicate: {mr["author_name"]}')
        continue

    rec = {
        'id': rid,
        'hotel': gr.ANANEA_HOTEL,
        'source': 'google',
        'rating': mr['rating'],
        'title': '',  # Google reviews have no title
        'text': mr['text'],
        'published_date': mr['published_date'],
        'author_name': mr['author_name'],
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
        'manual': True,
    }

    # Classify if Ollama available
    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')

    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"\u2705 {mr['author_name']} \u2192 {topic_str or '(unclassified)'}")

if added:
    gr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

✅ Seba Le → commodities(pos), quality_price(pos), employees(pos)
✅ Hans Naus → commodities(pos), meals(pos), comfort(pos), employees(pos), quality_price(pos)
✅ Heger → (unclassified)
✅ Ersen Unsurorlu → employees(pos)
✅ Pete → employees(pos), commodities(pos), quality_price(pos), meals(pos)
✅ Silke Engeln → employees(pos), meals(pos)

Saved 6 new reviews. Total: 33


## 5. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [10]:
json_path = str(ROOT / 'data' / 'google_reviews.json')
reviews = gr.load_reviews(json_path)

# Find reviews that are "classified" but have 0 topics \u2013 likely LLM failures
needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
# Also find unclassified reviews
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            author = r.get('author_name', 'N/A')
            print(f"  \u2705 {author[:40]} \u2192 {topic_str or '(still none)'}")
        except Exception as e:
            author = r.get('author_name', 'N/A')
            print(f"  \u274c {author[:40]} \u2192 Error: {e}")

    gr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

Total reviews: 33
Classified with 0 topics (needs retry): 4
Unclassified: 0
  ✅ K IKA → (still none)
  ✅ Larissa Goldnik → (still none)
  ✅ Christophe LEHMANN → employees(pos), commodities(pos)
  ✅ Heger → (still none)

Done. Saved 33 reviews.


## 6. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [8]:
json_path = str(ROOT / 'data' / 'google_reviews.json')
reviews = gr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '\U0001f504' if old_str != new_str else '\u2705'
            author = r.get('author_name', 'N/A')
            print(f"  {changed} [{i}/{len(with_text)}] {author[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            author = r.get('author_name', 'N/A')
            print(f"  \u274c [{i}/{len(with_text)}] {author[:40]} \u2192 Error: {e}")

    gr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

Total reviews: 33
Reviews with text (will reclassify): 33
  🔄 [1/33] Raj
       old: employees(pos), commodities(pos), comfort(neg), cleaning(pos), meals(pos), meals(neg), commodities(neg)
       new: employees(pos), commodities(pos), comfort(pos), meals(pos), cleaning(pos), comfort(neg), meals(neg), commodities(neg)
  🔄 [2/33] Lorena David
       old: employees(pos), commodities(pos), commodities(neg), meals(pos), meals(neg)
       new: employees(pos), commodities(pos), comfort(pos), cleaning(pos), commodities(neg), comfort(neg), meals(pos), meals(neg)
  ✅ [3/33] Elzbieta Krzyszton
  🔄 [4/33] Izzy Roberts
       old: employees(pos), commodities(pos)
       new: employees(pos)
  ✅ [5/33] Helen Turner
  🔄 [6/33] Anne b
       old: employees(pos), cleaning(pos), quality_price(pos)
       new: commodities(pos), employees(pos), quality_price(pos)
  🔄 [7/33] Anita
       old: (none)
       new: employees(pos), meals(pos), return(pos)
  🔄 [8/33] Alex Vl
       old: employees(pos), meals(pos)

## 7. Run CLI Scraper

Run the full CLI scraper via `python -m` to test the end-to-end pipeline.

In [ ]:
import subprocess
from datetime import datetime

date_col = datetime.now().strftime('%Y-%m-%d')
cmd = [sys.executable, '-m', 'src.sites.google_reviews',
       '--date', date_col, '--skip-classification']
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print(f'Return code: {result.returncode}')

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'google_reviews.json')
reviews = gr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if r.get("manual"))}')
print()

# Rating distribution
if reviews:
    ratings = [r.get('rating', 0) for r in reviews if r.get('rating')]
    if ratings:
        print(f'Average rating: {sum(ratings)/len(ratings):.1f}/5')
        for star in range(5, 0, -1):
            count = ratings.count(star)
            bar = '\u2588' * count
            print(f'  {star}\u2605: {bar} ({count})')
        print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')